# RecoverAI - Movement Quality Model

This notebook generates synthetic data for the "Arm Raise" exercise (simulating range of motion using the shoulder angle) and trains a simple PyTorch neural network to classify the movement as correct or incorrect. It then exports the model to ONNX so it can run directly in the browser.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import onnx

print(f"PyTorch version: {torch.__version__}")

## 1. Data Generation

A full repetition is modeled as 30 frames. The shoulder angle is tracked over these frames.
- **Correct Repetition**: Reaches an angle > 150 degrees.
- **Incorrect Repetition**: Reaches a peak angle < 120 degrees (poor range of motion).

In [ ]:
NUM_SAMPLES = 2000
FRAMES = 30
FEATURES = 1

def generate_data():
    X = np.zeros((NUM_SAMPLES, FRAMES, FEATURES), dtype=np.float32)
    y = np.zeros(NUM_SAMPLES, dtype=np.int64)
    
    for i in range(NUM_SAMPLES):
        is_correct = np.random.rand() > 0.5
        noise = np.random.normal(0, 5, FRAMES)
        
        if is_correct:
            peak = np.random.uniform(150, 180)
            y[i] = 1
        else:
            peak = np.random.uniform(60, 120)
            y[i] = 0
            
        x_vals = np.linspace(-3, 3, FRAMES)
        curve = np.exp(-x_vals**2) * peak
        X[i, :, 0] = np.clip(curve + noise, 0, 180)
        
    return X, y

X_data, y_data = generate_data()
X_tensor = torch.tensor(X_data)
y_tensor = torch.tensor(y_data)

print(f"Dataset generated: {X_tensor.shape}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(X_data[y_data == 1][0], label='Correct (High ROM)', color='green')
plt.plot(X_data[y_data == 0][0], label='Incorrect (Low ROM)', color='red')
plt.title('Synthetic Arm Raise Angles')
plt.xlabel('Frame')
plt.ylabel('Shoulder Angle (degrees)')
plt.legend()
plt.show()

## 2. Model Definition & Training
We use a simple Multi-Layer Perceptron (MLP) for this task since the input sequence length is fixed (30 frames).

In [ ]:
class RepetitionEvaluator(nn.Module):
    def __init__(self):
        super(RepetitionEvaluator, self).__init__()
        self.fc1 = nn.Linear(FRAMES * FEATURES, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2) # 2 classes: incorrect, correct
        
    def forward(self, x):
        x = x.view(x.size(0), -1) # Flatten (batch_size, frames)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = RepetitionEvaluator()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Training model...")
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

## 3. Export to ONNX
The web application uses ONNX Runtime Web to run inference directly in the browser.

In [ ]:
dummy_input = torch.randn(1, FRAMES, FEATURES)
onnx_path = "repetition_model.onnx"

torch.onnx.export(model, dummy_input, onnx_path, 
                  export_params=True,
                  opset_version=10, 
                  do_constant_folding=True,
                  input_names = ['input'], 
                  output_names = ['output'],
                  dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}})

print(f"Model successfully exported to {onnx_path}")